# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library. The dataset contains quantitative mass spectrometry data from extracellular vesicles (EVs) isolated from human mast cells, capturing fields such as protein accession, coverage, peptide counts, molecular weight, and normalized abundances for proteins in different samples.

### Dataset Source
The dataset source is provided via a [Croissant](https://mlcommons.org/croissant/) schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL as provided
CROISSANT_URL = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(CROISSANT_URL)
metadata = dataset.metadata

print('Name:', metadata.name)
print('Description:', metadata.description)
print('Version:', getattr(metadata, 'version', 'N/A'))


## 2. Data Overview
Explore available record sets, their identifiers (`@id`), and fields. All Croissant entities should be referenced by their `@id`.


In [ ]:
from pprint import pprint

# List available record sets by @id and descriptive info
print('Available Record Sets:')
record_sets = list(dataset.record_sets.values())
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            print(f"      - @id: {field.id}, name: {field.name}, dtype: {field.data_type}")
    print()

# Save the first record set's @id for use below
record_set_ids = [rs.id for rs in record_sets]
if record_set_ids:
    record_set_id = record_set_ids[0]
    print(f'Record set selected for exploration: {record_set_id}')
else:
    print('No record sets found!')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
dfs = {}
for rs_id in record_set_ids:
    print(f"Loading records for {rs_id} ...")
    records_iter = dataset.records(record_set=rs_id)
    # Collect records (as dict) and create a DataFrame
    records = list(records_iter)
    if len(records) > 0:
        dfs[rs_id] = pd.DataFrame(records)
        print(f"  {len(records)} records loaded into DataFrame.")
    else:
        print("  No records found.")
    print()

# Display extracted columns from the primary record set
if record_set_id in dfs:
    print(f"Columns in '{record_set_id}':")
    pprint(list(dfs[record_set_id].columns))
    display(dfs[record_set_id].head(3))
else:
    print(f"No DataFrame created for {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Let's filter and process the data.

- Select a numeric field (by `@id`) for filtering and normalization.
- Example: Filter out proteins with more than a threshold "peptide count".
- Normalize a numeric field (e.g., molecular weight).
- Group by a categorical field if available (e.g., sample group, protein type).

In [ ]:
# Choose a numeric field based on available columns (@id should be used; adjust as needed)
primary_df = dfs[record_set_id]

# Try common column names: Find a numeric field
candidate_numeric_fields = [
    col for col in primary_df.columns
    if ('count' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower()
       or 'coverage' in col.lower() or ('peptide' in col.lower() and 'id' not in col.lower()))
    and pd.api.types.is_numeric_dtype(primary_df[col])
]
if not candidate_numeric_fields:
    candidate_numeric_fields = [col for col in primary_df.columns if pd.api.types.is_numeric_dtype(primary_df[col])]

if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
    print(f"Selected numeric field (by @id or column): {numeric_field_id}")
else:
    raise ValueError('No numeric field found in dataset.')

# Set filter threshold
threshold = primary_df[numeric_field_id].quantile(0.75) if len(primary_df) > 10 else 10

filtered_df = primary_df[primary_df[numeric_field_id] > threshold].copy()
print(f"Filtered to records where {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)} records)")
display(filtered_df.head())

# Normalize the numeric field (z-score)
normalized_field = f"{numeric_field_id}_normalized"
filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' (z-score) for filtered records.")
display(filtered_df[[numeric_field_id, normalized_field]].head())

# Attempt groupby by a suitable categorical field (by @id)
candidate_group_fields = [
    col for col in filtered_df.columns
    if col not in candidate_numeric_fields and pd.api.types.is_object_dtype(filtered_df[col])
    and filtered_df[col].nunique() < filtered_df.shape[0]//2
    and filtered_df[col].nunique() > 1
]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    print(f"Grouping by categorical field (by @id): {group_field_id}\n")
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean of '{numeric_field_id}' by group '{group_field_id}':")
    display(grouped.head())
else:
    print('No suitable group-by field found, skipping grouping.')


## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, by categorical group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered records)")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group if grouping variable available
if 'group_field_id' in locals():
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()


## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR² mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library:

- Loaded metadata and discovered record sets using the Croissant schema.
- Extracted records into Pandas DataFrames and explored available fields and their `@id`s.
- Filtered and normalized a numeric variable by referencing columns via `@id`.
- Previewed data distributions and summarized values by group, where available.

You can continue analysis for downstream machine learning, statistics, or visualization tasks as needed.

---
For more information on the dataset structure, see the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant).
